In [1]:
from huggingface_hub import notebook_login

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
notebook_login()

In [3]:
from datasets import load_dataset

In [4]:
dataset = load_dataset("eriktks/conll2003",revision="convert/parquet")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [6]:
df=dataset["train"].to_pandas()

In [7]:
df.head()

,id,tokens,pos_tags,chunk_tags,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."


In [8]:
ner_feature_names=dataset["train"].features["ner_tags"]

In [9]:
label_names=ner_feature_names.feature.names

In [10]:
####If you look closely, the B-XXX labels have an odd index (label%2==1)

In [11]:
chunk_feature_names=dataset["train"].features['chunk_tags']

In [12]:
chunk_feature_names.feature.names

['O',
 'B-ADJP',
 'I-ADJP',
 'B-ADVP',
 'I-ADVP',
 'B-CONJP',
 'I-CONJP',
 'B-INTJ',
 'I-INTJ',
 'B-LST',
 'I-LST',
 'B-NP',
 'I-NP',
 'B-PP',
 'I-PP',
 'B-PRT',
 'I-PRT',
 'B-SBAR',
 'I-SBAR',
 'B-UCP',
 'I-UCP',
 'B-VP',
 'I-VP']

In [13]:
pos_feature_names=dataset["train"].features["pos_tags"]

In [14]:
pos_feature_names.feature.names

['"',
 "''",
 '#',
 '$',
 '(',
 ')',
 ',',
 '.',
 ':',
 '``',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'NN|SYM',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB']

In [15]:
from transformers import AutoModel,AutoTokenizer,AutoModelForTokenClassification

In [16]:
checkpoint="bert-base-cased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

In [17]:
inputs=tokenizer(dataset["train"][0]['tokens'],is_split_into_words=True)

In [18]:
inputs

{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [19]:
inputs.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [20]:
inputs.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [21]:
def align_labels_with_word_ids(labels,word_ids):
    new_labels=[]
    current_word=None
    for word_id in word_ids:
        if word_id!=current_word:
            current_word=word_id
            label=-100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label=labels[word_id]
            if label%2==1:
                label+=1
            new_labels.append(label)
    return new_labels

In [22]:
labels=dataset['train'][0]['ner_tags']
word_ids=inputs.word_ids()
align_labels_with_word_ids(labels,word_ids)

[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]

In [23]:
def tokenize_and_align_labels(dataset):
    tokenized_dataset=tokenizer(dataset['tokens'],is_split_into_words=True,truncation=True)
    all_labels=dataset['ner_tags']
    new_labels=[]
    for i,label in enumerate(all_labels):
        word_ids=tokenized_dataset.word_ids(batch_index=i)
        new_labels.append(align_labels_with_word_ids(label,word_ids))
    tokenized_dataset['labels']=new_labels
    return tokenized_dataset

In [24]:
tokenized_dataset=dataset.map(tokenize_and_align_labels,batched=True,remove_columns=dataset['train'].column_names)

In [25]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [26]:
from transformers import DataCollatorForTokenClassification

In [27]:
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)


In [28]:
import evaluate
metric=evaluate.load('seqeval')

In [29]:
metric

EvaluationModule(name: "seqeval", module_type: "metric", features: {'predictions': List(Value('string')), 'references': List(Value('string'))}, usage: """
Produces labelling scores along with its sufficient statistics
from a source against one or more references.

Args:
    predictions: List of List of predicted labels (Estimated targets as returned by a tagger)
    references: List of List of reference labels (Ground truth (correct) target values)
    suffix: True if the IOB prefix is after type, False otherwise. default: False
    scheme: Specify target tagging scheme. Should be one of ["IOB1", "IOB2", "IOE1", "IOE2", "IOBES", "BILOU"].
        default: None
    mode: Whether to count correct entity labels with incorrect I/B tags as true positives or not.
        If you want to only count exact matches, pass mode="strict". default: None.
    sample_weight: Array-like of shape (n_samples,), weights for individual samples. default: None
    zero_division: Which value to substitute as a

In [41]:
import numpy as np

def compute_metrics(eval_preds):
    logits,labels=eval_preds
    predictions=np.argmax(logits,axis=-1)
    true_labels=[[label_names[i] for i in label if i!=-100] for label in labels]
    true_predictions=[[label_names[p] for (p,l) in zip(prediction,label) if l!=-100] for prediction,label in zip(predictions,labels)]
    all_metrics=metric.compute(predictions=true_predictions,references=true_labels)

    return {
        "accuracy":all_metrics["overall_accuracy"],
        "precision":all_metrics["overall_precision"],
        "recall":all_metrics["overall_recall"],
        "f1 score":all_metrics["overall_f1"]
        
    }

In [31]:
id2label={i:label for i,label in enumerate(label_names)}
label2id={v:k for k,v in id2label.items()}

In [32]:
model=AutoModelForTokenClassification.from_pretrained(checkpoint,id2label=id2label,label2id=label2id)

Loading weights: 100%|████████████████████| 197/197 [00:00<00:00, 24622.38it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archite

In [33]:
model.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC",
    "7": "B-MISC",
    "8": "I-MISC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B-LOC": 5,
    "B-MISC": 7,
    "B-ORG": 3,
    "B-PER": 1,
    "I-LOC": 6,
    "I-MISC": 8,
    "I-ORG": 4,
    "I-PER": 2,
    "O": 0
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,


In [38]:
from transformers import TrainingArguments,Trainer

In [36]:
args=TrainingArguments("bert-finetuned-ner",learning_rate=2e-5,eval_strategy="epoch",save_strategy="epoch",num_train_epochs=3,weight_decay=0.01,push_to_hub=True)

In [42]:
trainer=Trainer(model=model,args=args,train_dataset=tokenized_dataset["train"],eval_dataset=tokenized_dataset["validation"],data_collator=data_collator,compute_metrics=compute_metrics,processing_class=tokenizer)

In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 score
1,0.023814,0.068692,0.985268,0.928961,0.946314,0.937557
2,0.018921,0.065271,0.986740,0.936763,0.949849,0.943261
3,0.011309,0.065037,0.987226,0.939947,0.953551,0.946700


Writing model shards: 100%|██████████████████████| 1/1 [00:00<00:00,  2.09it/s]
/opt/anaconda3/envs/tf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████| 1/1 [00:00<00:00,  2.69it/s]
/opt/anaconda3/envs/tf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████| 1/1 [00:00<00:00,  2.58it/s]


TrainOutput(global_step=5268, training_loss=0.02397906698778713, metrics={'train_runtime': 2106.4728, 'train_samples_per_second': 19.997, 'train_steps_per_second': 2.501, 'total_flos': 920771584279074.0, 'train_loss': 0.02397906698778713, 'epoch': 3.0})

In [44]:
from transformers import pipeline

In [45]:
model_checkpoint="Adnan2942/bert-finetuned-ner"
token_classifier=pipeline("token-classification",model=model_checkpoint,aggregation_strategy="simple")

Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 20666.71it/s]


In [64]:
token_classifier("Adnan Iqbal is learning by doing")

[{'entity_group': 'PER',
  'score': 0.9997441,
  'word': 'Adnan Iqbal',
  'start': 0,
  'end': 11}]